# 🏭 Chest X-Ray — Tüm Çıktıları Üret (Downstream Production Notebook)

Bu notebook **modelleri YENİDEN EĞİTMEZ**. Eğitilmiş ağırlıklar Drive'daki
`xray_outputs` içinden geri yüklenir; bu oturum yalnızca **eksik ara/sonuç
verilerini** üretir:

- per-image tahminler (`tiered_predictions.csv`, Ark+ ve EfficientNet)
- A13 (tiered) ve A14 (CheXpert zero-shot) değerlendirme JSON'ları (eksikse)
- gerçek `ablation.json` (uydurma yok), istatistik testleri, model karşılaştırması
- tez figürleri + 5 analiz notebook'u (gerçek tahminlerden)
- ONNX export + gecikme/karbon benchmark'ı
- (opsiyonel) Stable Diffusion sentetik veri

**Tasarım ilkeleri:**
1. Her adımın `run` (çalıştır) ve `force` (varsa bile üzerine yaz) anahtarı var → `STEPS`.
2. `SKIP_EXISTING=True` iken zaten üretilmiş çıktılar atlanır (kısmi hata sonrası
   yeniden çalıştırınca baştan başlamaz). `FORCE_ALL=True` her şeyi yeniden üretir.
3. Hiçbir hücre çökmez: her adım try/except içinde, sonunda durum raporu basılır.
4. Drive'a kayıt **tek `xray_outputs/outputs` ağacına MERGE** eder; baz model
   ağırlıkları (.pth) yeniden yüklenmez, tarihli yeni zip kopyaları üretilmez.


## 0. Yapılandırma ve Anahtarlar

In [ ]:
# ============================================================================
# ⚙️ YAPILANDIRMA — produce-all (downstream) pipeline
# ============================================================================
import shutil
shutil.rmtree('/content/sample_data', ignore_errors=True)

GITHUB_REPO_URL  = 'https://github.com/xdboi123yes/xray.git'
GITHUB_BRANCH    = 'main'
PROJECT_ROOT     = '/content/xray'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/xray_outputs'
USE_DRIVE        = True
KAGGLE_DATASET   = 'nih-chest-xrays/data'
DRY_RUN          = False

# --- Global switches -------------------------------------------------------
SKIP_EXISTING    = True    # skip a step if its outputs already exist
FORCE_ALL        = False   # regenerate EVERYTHING, overwriting outputs (overrides SKIP_EXISTING)

# --- Compute knobs ---------------------------------------------------------
N_BOOT           = 1000    # bootstrap iterations for statistical tests
MC_PASSES        = 20      # MC-dropout passes for prediction export

# --- Drive save behaviour --------------------------------------------------
SAVE_TO_DRIVE      = True    # merge this session's NEW files into DRIVE/xray_outputs/outputs
MAKE_ZIP_SNAPSHOT  = False   # also write ONE results zip (same path, overwritten — never dated copies)
BIG_FILE_MB        = 50      # files larger than this that were NOT produced now are treated as base
                             # weights and never re-uploaded to Drive

# --- Per-step switches: run = execute, force = overwrite even if outputs exist ---
# Toggle any single model/file independently here.
STEPS = {
    # NOTE: setup cells (Drive restore, NIH cache, CheXpert download, conformal q_hat)
    # are idempotent and self-skip via their own cache checks, so they are not listed
    # here. Every key below maps to exactly one produce() step in the cells.
    # --- Model training: SKIPS if weights already exist (restored from Drive).
    #     Set force=True (or FORCE_ALL, or delete the .pth) to retrain & overwrite. ---
    'train_tier1':              {'run': True,  'force': False},
    'train_tier2_efficientnet': {'run': True,  'force': False},
    'train_tier2_arkplus':      {'run': True,  'force': False},
    'ablation_A8':              {'run': True,  'force': False},
    'ablation_A9':              {'run': True,  'force': False},
    'ablation_A11':             {'run': True,  'force': False},
    'ablation_A12':             {'run': True,  'force': False},
    'ablation_A15':             {'run': True,  'force': False},
    'calibration_temperature':  {'run': True,  'force': False},
    'eval_A13_tiered':          {'run': True,  'force': False},
    'eval_A14_chexpert':        {'run': True,  'force': False},
    'predictions_arkplus':      {'run': True,  'force': False},
    'predictions_efficientnet': {'run': True,  'force': False},
    'figure_input_csvs':        {'run': True,  'force': False},
    'ablation_json':            {'run': True,  'force': False},
    'statistical_tests':        {'run': True,  'force': False},
    'compare_models':           {'run': True,  'force': False},
    'report_figures':           {'run': True,  'force': False},
    'nb_subgroup_analysis':     {'run': True,  'force': False},
    'nb_decision_curve':        {'run': True,  'force': False},
    'nb_error_analysis':        {'run': True,  'force': False},
    'nb_tier_disagreement':     {'run': True,  'force': False},
    'nb_synthetic_quality':     {'run': True,  'force': False},
    'onnx_tier1':               {'run': True,  'force': False},
    'onnx_tier2':               {'run': True,  'force': False},
    'benchmark_latency':        {'run': True,  'force': False},
    'synthetic_generation':     {'run': False, 'force': False},  # heavy SD; off by default
}

print('✅ Config loaded.')
print(f'  • SKIP_EXISTING={SKIP_EXISTING}  FORCE_ALL={FORCE_ALL}')
print(f'  • Drive merge → {DRIVE_OUTPUT_DIR}/outputs  (zip snapshot={MAKE_ZIP_SNAPSHOT})')
print(f'  • {sum(1 for s in STEPS.values() if s["run"])}/{len(STEPS)} steps enabled')


## 1. Ortam Tanılama & GPU

In [ ]:
import os
import sys

import torch

print(f'🐍 Python version: {sys.version.split()[0]}')
print(f'🔥 PyTorch version: {torch.__version__}')
print(f'⚡ CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'   🚀 GPU:   {device_name}')
    print(f'   💾 VRAM:  {total_vram:.1f} GB')
    if total_vram > 30:
        print('   ✨ Excellent! Detected high-end A100 GPU. Optimum batch loading and tensor parallelism are active.')
else:
    print('   ⚠️  No GPU detected. Under Runtime > Change runtime type, select a GPU accelerator.')

## 2. Google Drive Bağla

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    print('📂 Mounting Google Drive...')
    drive.mount('/content/drive')
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print(f'✅ Drive mounted! Outputs will sync to: {DRIVE_OUTPUT_DIR}')
else:
    print('⏭️ Google Drive mount skipped.')

## 3. Repo Klonla & Bağımlılıkları Kur

In [ ]:
import subprocess
import os
import sys

# Capture Colab's pre-installed NumPy version BEFORE touching the environment.
# Colab's scientific stack (pandas / scipy / scikit-learn / torch) is ABI-compiled against
# this exact NumPy. The project's older pinned deps can silently drag NumPy to a different
# version during the install below, which then breaks pandas the next time it is imported:
#   ValueError: numpy.dtype size changed, may indicate binary incompatibility.
#   Expected 96 from C header, got 88 from PyObject
# We restore Colab's NumPy right after installing so the ABI stays consistent and the kernel
# is never poisoned.
try:
    import numpy as _np
    _COLAB_NUMPY = _np.__version__
except Exception:
    _COLAB_NUMPY = None

if not os.path.exists(PROJECT_ROOT):
    print(f'📥 Cloning repository from {GITHUB_REPO_URL}...')
    subprocess.check_call(['git', 'clone', '-b', GITHUB_BRANCH, GITHUB_REPO_URL, PROJECT_ROOT])
else:
    print(f'📂 Repository already exists at {PROJECT_ROOT}. Pulling latest updates...')
    os.chdir(PROJECT_ROOT)
    subprocess.call(['git', 'pull'])

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print('\n📦 Installing required packages (this may take ~1 min)...')
packages = [
    'timm==0.9.16',
    'albumentations==1.4.3',
    'mlflow>=2.10.0',
    'structlog',
    'gdown',
    'onnx>=1.15.0',
    'onnxruntime>=1.17.1',
    'tqdm>=4.66.0',
    'grad-cam==1.5.0',
    'pytorch-fid==0.3.0',
    'gradio==4.21.0',
    'plotly==5.19.0',
    'nonconformist==2.1.0',
    'memory-profiler==0.61.0',
    'diffusers==0.27.2',
    'transformers==4.38.2',
    'accelerate==0.28.0',
    'fastapi==0.110.0',
    'uvicorn==0.28.0',
    'websockets>=10.0,<12.0',
    'pydantic==2.6.4',
    'pydantic-settings==2.2.1',
    'import-linter==2.0',
    'slowapi==0.1.9',
    'optuna>=3.6'
]

try:
    print('⚡ Attempting standard requirements installation...')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        '-r', 'requirements.txt',
        '-r', 'requirements-training.txt'
    ])
except Exception as e:
    print(f'\n⚠️ Standard installation encountered conflicts: {e}')
    print('🔄 Activating adaptive fallback (preserving Colab\'s optimized PyTorch/CUDA environment for safety)...')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', *packages
    ])

# --- NumPy ABI guard -------------------------------------------------------
# Restore Colab's original NumPy so the pre-built pandas / scipy / sklearn / torch ABI is intact.
if _COLAB_NUMPY:
    print(f'\n🔧 Re-pinning NumPy to Colab\'s pre-built version ({_COLAB_NUMPY}) to avoid ABI breakage...')
    _restore = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', f'numpy=={_COLAB_NUMPY}'],
        capture_output=True, text=True
    )
    if _restore.returncode != 0:
        print('   ⚠️ Could not pin the exact version; falling back to the NumPy 2.x line...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'numpy>=2.0'])

# Verify the scientific stack imports cleanly in a FRESH process. The running kernel may still
# hold a stale NumPy in memory; if this check fails, a one-time runtime restart fixes it (your
# /content data is preserved and the Kaggle download is auto-skipped on re-run).
_check = subprocess.run(
    [sys.executable, '-c', 'import numpy, pandas; print(numpy.__version__, pandas.__version__)'],
    capture_output=True, text=True
)
if _check.returncode == 0:
    print(f'✅ Scientific stack OK in a fresh process (numpy / pandas = {_check.stdout.strip()}).')
else:
    _last = (_check.stderr.strip().splitlines() or ['unknown error'])[-1]
    print('⚠️ NumPy / pandas ABI mismatch still present:')
    print('   ' + _last[:200])
    print('   → Runtime > Restart session, then Run all. /content data is preserved; the dataset download auto-skips.')

print('✅ Codebase synced and all dependencies configured successfully!')

## 4. Üretim Motoru (skip/force, merge-save altyapısı)

In [ ]:
# ============================================================================
# 🛠️ ÜRETİM MOTORU: idempotent, asla çökmeyen adım yürütücü
# ============================================================================
import os, sys, subprocess, traceback, time

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

PRODUCED = set()    # absolute paths created/refreshed in THIS session
RUN_REPORT = []     # (step, status, detail)

# NOTE: named _step_cfg (not _cfg) on purpose — a reused setup cell uses `_cfg`
# as a loop variable, which would otherwise shadow this helper in the shared
# notebook namespace and make produce() raise "'dict' object is not callable".
def _step_cfg(name):
    c = STEPS.get(name, {'run': True, 'force': False})
    return bool(c.get('run', True)), (bool(c.get('force', False)) or FORCE_ALL)

def _exist_all(paths):
    return bool(paths) and all(os.path.exists(p) for p in paths)

def _log(step, status, detail=''):
    icon = {'DONE':'✅','SKIPPED':'⏭️','DISABLED':'🚫','BLOCKED':'⛔','PARTIAL':'⚠️','ERROR':'❌'}.get(status, '•')
    print(f'{icon} [{status}] {step}' + (f' — {detail}' if detail else ''))
    RUN_REPORT.append((step, status, str(detail)))

def run_script(args, env=None):
    e = dict(os.environ)
    if env:
        e.update(env)
    print('   $ ' + ' '.join(str(a) for a in args))
    proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=e)
    for line in proc.stdout:
        print('   | ' + line, end='')
    return proc.wait()

def sh(args, env=None):
    rc = run_script(args, env)
    if rc != 0:
        raise RuntimeError(f'command failed (exit {rc}): ' + ' '.join(str(a) for a in args))

def produce(name, outputs=None, fn=None, inputs=None, always=False):
    '''Run step `name` defensively.

    - run=False            -> DISABLED
    - missing inputs       -> BLOCKED (no throw)
    - outputs exist + skip -> SKIPPED (unless force / always / FORCE_ALL)
    - else run fn()        -> DONE / PARTIAL / ERROR (never raises out of this cell)
    Output paths of a successful run are recorded in PRODUCED (drives Drive save).
    '''
    outputs = outputs or []
    inputs = inputs or []
    run, force = _step_cfg(name)
    if not run:
        _log(name, 'DISABLED'); return
    missing = [p for p in inputs if not os.path.exists(p)]
    if missing:
        _log(name, 'BLOCKED', f'missing inputs: {missing}'); return
    if SKIP_EXISTING and not force and not always and _exist_all(outputs):
        _log(name, 'SKIPPED', 'outputs present'); return
    t0 = time.time()
    try:
        if fn is not None:
            fn()
        if outputs and not _exist_all(outputs):
            miss = [p for p in outputs if not os.path.exists(p)]
            _log(name, 'PARTIAL', f'ran but missing: {miss}'); return
        for p in outputs:
            PRODUCED.add(os.path.abspath(p))
        _log(name, 'DONE', f'{time.time() - t0:.1f}s')
    except Exception as exc:
        traceback.print_exc()
        _log(name, 'ERROR', repr(exc))

print('✅ Engine ready: produce(name, outputs, fn, inputs, always).')


## 5. Kaggle Kimlik Bilgileri (NIH/CheXpert indirme için)

In [ ]:
import os, json
from pathlib import Path

KAGGLE_CONFIG_DIR = Path.home() / '.kaggle'
KAGGLE_CONFIG_DIR.mkdir(exist_ok=True)
KAGGLE_JSON_PATH = KAGGLE_CONFIG_DIR / 'kaggle.json'

# @markdown ### 🌐 Option A: Copy-Paste Kaggle API Credentials (Recommended)
# @markdown Copy-paste your Kaggle username and API key directly from your Kaggle settings:
KAGGLE_USERNAME = "" # @param {type:"string"}
KAGGLE_KEY = "" # @param {type:"string"}

if KAGGLE_USERNAME and KAGGLE_KEY:
    with open(KAGGLE_JSON_PATH, 'w') as f:
        json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
    KAGGLE_JSON_PATH.chmod(0o600)
    print(f"✅ Kaggle credentials saved from form fields! User: {KAGGLE_USERNAME}")
elif KAGGLE_JSON_PATH.exists():
    print(f"✅ Kaggle credentials already present at: {KAGGLE_JSON_PATH}")
else:
    # @markdown ---
    # @markdown ### 📤 Option B: Direct File Upload
    # @markdown If form fields are empty, upload your downloaded `kaggle.json` file here:
    try:
        from google.colab import files
        print("📤 Upload your kaggle.json:")
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            Path('kaggle.json').rename(KAGGLE_JSON_PATH)
            KAGGLE_JSON_PATH.chmod(0o600)
            print("✅ Kaggle credentials uploaded and installed successfully!")
        else:
            print("⏭️ Upload skipped or incomplete.")
    except Exception as e:
        print(f"⏭️ Direct upload skipped (non-interactive environment): {e}")

## 6. Modelleri & Önceki Çıktıları Geri Yükle (yeniden indirme/zip YOK)

In [ ]:
import os
import sys
import shutil
import zipfile
import glob

print('🔍 Scanning for pre-trained weights to restore...')

models_dir = os.path.join(PROJECT_ROOT, 'outputs', 'models')
outputs_dir = os.path.join(PROJECT_ROOT, 'outputs')
os.makedirs(models_dir, exist_ok=True)

# 1. ZIP File Restoration (Uploaded directly or in Drive) — ALL archives, not just the first.
zip_candidates = glob.glob('/content/*.zip')
if USE_DRIVE and os.path.exists('/content/drive/MyDrive'):
    zip_candidates.extend(glob.glob('/content/drive/MyDrive/*.zip'))
    zip_candidates.extend(glob.glob('/content/drive/MyDrive/xray*/*.zip'))

# De-duplicate (a file can match several globs) while preserving order.
_seen = set()
unique_zips = []
for _z in zip_candidates:
    _rp = os.path.realpath(_z)
    if _rp not in _seen:
        _seen.add(_rp)
        unique_zips.append(_z)


def _zip_extract_target(names):
    """Pick the correct extraction root from a ZIP's member layout.

    * Archive already contains an ``outputs/`` tree         -> extract to PROJECT_ROOT.
    * Archive root *is* the ``outputs/`` dir (top-level      -> extract to outputs/.
      ``models/`` ``results/`` ``figures/`` members, as
      produced by ``shutil.make_archive(root_dir=outputs)``)
    * Bare ``*.pth`` weight files                            -> extract to outputs/models/.
    """
    def has_top(prefix):
        return any(n == prefix or n.startswith(prefix) or n.startswith('./' + prefix) for n in names)
    if has_top('outputs/'):
        return PROJECT_ROOT
    if has_top('models/') or has_top('results/') or has_top('figures/'):
        return outputs_dir
    return models_dir


print(f'📦 Found {len(unique_zips)} potential ZIP file(s) for restoration:')
for idx, path in enumerate(unique_zips):
    print(f'  [{idx}] {path} ({os.path.getsize(path) / 1e6:.2f} MB)')

zip_extracted = False
for target_zip in unique_zips:
    print(f'\n⚡ Extracting weights from ZIP: {target_zip}...')
    try:
        with zipfile.ZipFile(target_zip, 'r') as zip_ref:
            target = _zip_extract_target(zip_ref.namelist())
            print(f'  📂 Extracting into: {target}')
            zip_ref.extractall(target)
        print('  ✅ ZIP extraction complete!')
        zip_extracted = True
    except Exception as e:
        print(f'  ❌ Error during ZIP extraction: {e}')
if not unique_zips:
    print('\n⏭️ No ZIP archives found.')

# 2. Google Drive Deep Subfolder Recovery (e.g. xray_notebooks/<timestamp>/)
recovered_from_drive = []
if USE_DRIVE and os.path.exists('/content/drive/MyDrive'):
    print('\n📂 Drive mounted. Scanning xray_notebooks/ and xray_outputs/ subfolders recursively for trained weights...')
    # Search patterns covering timestamped directories and arbitrary subfolders
    search_patterns = [
        '/content/drive/MyDrive/xray_notebooks/**/best_*.pth',
        '/content/drive/MyDrive/xray_notebooks/**/*.pth',
        '/content/drive/MyDrive/xray_outputs/**/*.pth',
        '/content/drive/MyDrive/best_*.pth',
    ]
    for pattern in search_patterns:
        matches = glob.glob(pattern, recursive=True)
        for m in matches:
            basename = os.path.basename(m)
            dst_path = os.path.join(models_dir, basename)
            if not os.path.exists(dst_path):
                shutil.copy2(m, dst_path)
                recovered_from_drive.append(m)
                print(f'  ✨ Automatically recovered weight file: {basename} (from {m})')

if recovered_from_drive:
    print(f'🎉 Recovered {len(recovered_from_drive)} weight files directly from Google Drive folders!')

# 3. Filename Alignments and Alt Conventions
weight_files = glob.glob(os.path.join(models_dir, '**/*.pth'), recursive=True)
for wf in weight_files:
    basename = os.path.basename(wf)
    if basename == 'best_tier2_arkplus.pth' and not os.path.exists(os.path.join(models_dir, 'best_tier2_ark_plus.pth')):
        shutil.copy2(wf, os.path.join(models_dir, 'best_tier2_ark_plus.pth'))
        print('  🔄 Aligned: best_tier2_arkplus.pth → best_tier2_ark_plus.pth')
    elif basename == 'best_tier2_ark_plus.pth' and not os.path.exists(os.path.join(models_dir, 'best_tier2_arkplus.pth')):
        shutil.copy2(wf, os.path.join(models_dir, 'best_tier2_arkplus.pth'))
        print('  🔄 Aligned: best_tier2_ark_plus.pth → best_tier2_arkplus.pth')

# 4. Check for standard weights and show status
required_files = [
    'best_tier1_mobilenet.pth',
    'best_tier2_efficientnet.pth',
    'best_tier2_ark_plus.pth'
]

print('\n⚙️ Core Checkpoint Recovery Status Report:')
all_found = True
for rf in required_files:
    target_path = os.path.join(models_dir, rf)
    if os.path.exists(target_path):
        print(f'  ✅ [Restored] {rf} ({os.path.getsize(target_path) / 1e6:.2f} MB)')
    else:
        nested_matches = glob.glob(os.path.join(models_dir, f'**/{rf}'), recursive=True)
        if nested_matches:
            shutil.copy2(nested_matches[0], target_path)
            print(f'  ✅ [Found & Relocated] {rf} from nested subdirectory.')
        else:
            print(f'  ❌ [Missing]  {rf}')
            all_found = False

if all_found:
    print('\n🎉 ALL core weights recovered! Ready for zero-shot (A14) and tiered evaluation (A13) ablations.')
else:
    print('\n⚠️ Note: Some core model weights are missing. Training ablations (A8, A9, A11, A12, A15) can still be trained from scratch, but evaluation-only ablations (A13, A14) will fall back to initialized weights or raise warnings unless they are uploaded.')

# 5. Per-ablation completion status — drives the skip-existing logic in the training cell (#8).
#    Reuses the exact same detection functions the runner uses, so what is reported here as
#    "SKIP" is precisely what will be skipped, and "WILL RUN" is what still consumes GPU time.
try:
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)
    from application.orchestration.ablation_runner import AblationRunner
    from scripts.run_ablation import completion_marker, is_complete

    _runner = AblationRunner()
    print('\n🧭 Ablation completion status (this feeds SKIP_EXISTING in the training cell):')
    _done, _todo = [], []
    for _e in ['A8', 'A9', 'A11', 'A12', 'A13', 'A14', 'A15']:
        _cfg = _runner.ablation_configs[_e]
        if is_complete(_cfg):
            _done.append(_e)
            print(f'  ✅ {_e:<4} complete → SKIP      ({completion_marker(_cfg)})')
        else:
            _todo.append(_e)
            print(f'  ⬜ {_e:<4} missing  → WILL RUN  ({completion_marker(_cfg)})')
    print(f'\n  → {len(_done)} already done {_done}  |  {len(_todo)} pending {_todo}')
    if not _todo:
        print('  🎉 Every ablation already has results — the training cell will be a no-op.')
    elif SKIP_EXISTING:
        print(f'  ⏱️  With SKIP_EXISTING=True, only these will train/evaluate: {_todo}')
    else:
        print('  ⚠️  SKIP_EXISTING=False → every ablation will be re-run regardless of restored weights.')
except Exception as _status_err:
    print(f'  ⚠️ Could not compute ablation completion status: {_status_err}')

## 7. NIH Görüntü Önbelleği + Split'ler

In [ ]:
import glob
import os
import shutil
import subprocess

DRIVE_DATA   = '/content/drive/MyDrive/nih_data'
COLAB_DATA   = '/content/nih_data'
MERGED_DIR   = f'{COLAB_DATA}/images'
DRIVE_MERGED = f'{DRIVE_DATA}/images'
CSV_NAME     = 'Data_Entry_2017.csv'
MIN_IMAGES   = 50000

os.makedirs(COLAB_DATA, exist_ok=True)

def count_files(d):
    if not os.path.isdir(d): return 0
    return len([f for f in os.listdir(d) if os.path.isfile(os.path.join(d, f))])

# Check for pre-existing images
n_colab = count_files(MERGED_DIR)
if n_colab >= MIN_IMAGES:
    print(f'✅ Images found locally in Colab: {n_colab} files.')
elif count_files(DRIVE_MERGED) >= MIN_IMAGES:
    print('📂 Copying cached images from Google Drive to local disk (~5 mins)...')
    os.makedirs(MERGED_DIR, exist_ok=True)
    # Fast copy using shell utility
    subprocess.check_call(['cp', '-r', f'{DRIVE_MERGED}/.', MERGED_DIR])
    print(f'   ✅ Copied {count_files(MERGED_DIR)} files!')
else:
    print(f'🌐 Downloading fresh dataset from Kaggle: {KAGGLE_DATASET}...')
    if not os.path.exists(KAGGLE_JSON_PATH):
        print('⚠️ Error: kaggle.json credentials are required to download the dataset. Please upload it in step 5.')
    else:
        os.environ['KAGGLE_CONFIG_DIR'] = str(Path.home() / '.kaggle')
        subprocess.check_call([
            'kaggle', 'datasets', 'download',
            '-d', KAGGLE_DATASET,
            '-p', COLAB_DATA,
            '--unzip'
        ])
        # Merge nested folders if needed
        print('🔧 Flattening directories...')
        os.makedirs(MERGED_DIR, exist_ok=True)
        for i in range(1, 13):
            sub = os.path.join(COLAB_DATA, f'images_{i:03d}', 'images')
            if os.path.isdir(sub):
                for fname in os.listdir(sub):
                    shutil.move(os.path.join(sub, fname), os.path.join(MERGED_DIR, fname))
                shutil.rmtree(os.path.join(COLAB_DATA, f'images_{i:03d}'))
        print(f'   ✅ Merged: {count_files(MERGED_DIR)} images.')

# Check for split files
csv_dst = os.path.join(PROJECT_ROOT, 'data', 'raw', CSV_NAME)
os.makedirs(os.path.dirname(csv_dst), exist_ok=True)
for csv_candidate in [os.path.join(COLAB_DATA, CSV_NAME), os.path.join(DRIVE_DATA, CSV_NAME)]:
    if os.path.exists(csv_candidate):
        shutil.copy2(csv_candidate, csv_dst)
        print(f'✅ Found CSV metadata: {csv_candidate}')
        break

IMAGE_DIR = MERGED_DIR
os.makedirs(os.path.join(PROJECT_ROOT, 'data', 'processed'), exist_ok=True)
with open(os.path.join(PROJECT_ROOT, 'data', 'processed', 'image_dir.txt'), 'w') as f:
    f.write(IMAGE_DIR)

print('📊 Preprocessing splits...')
os.chdir(PROJECT_ROOT)
subprocess.check_call([sys.executable, 'scripts/preprocess.py', '--image-dir', IMAGE_DIR])

# Create calibration split if missing.
# NOTE: this runs in a SUBPROCESS (not in-kernel) on purpose. A fresh process always has a
# consistent NumPy/pandas ABI, so this step cannot be broken by any stale NumPy the kernel may
# still hold in memory — the same reason preprocess.py above runs cleanly.
calib_csv = os.path.join(PROJECT_ROOT, 'data', 'processed', 'calibration.csv')
if not os.path.exists(calib_csv):
    _split_code = (
        "import pandas as pd;"
        "v=pd.read_csv('data/processed/val.csv');"
        "v.sample(frac=0.3, random_state=42).to_csv('data/processed/calibration.csv', index=False);"
        "print('rows:', len(pd.read_csv('data/processed/calibration.csv')))"
    )
    subprocess.check_call([sys.executable, '-c', _split_code])
    print('  ✅ Generated calibration split (subprocess): data/processed/calibration.csv')


## 8. Model Eğitimi — Base (Tier1 / Tier2-Eff / Tier2-Ark) + Ablation (A8/A9/A11/A12/A15)

> Ağırlık zaten varsa (Drive'dan restore) bu adımlar **ATLANIR**. Belirli bir modeli yeniden eğitip üzerine yazmak için `STEPS['train_...']['force']=True` (ya da `FORCE_ALL=True`). `DRY_RUN=True` → model başına 1 epoch.

In [ ]:
# ----------------------------------------------------------------------------
# MODEL EĞİTİMİ (xray_colab_training_auto.ipynb'den port edildi).
# Her model, ağırlığı zaten varsa (Drive'dan restore) ATLAR. Yeniden eğitip
# üzerine yazmak için STEPS['<step>']['force'] = True (ya da FORCE_ALL, ya da
# .pth'i sil). DRY_RUN=True → model başına 1 epoch (hızlı uçtan uca test).
# Eğitim izole bir alt süreçte koşar (temiz GPU belleği + birebir hiperparametre).
# ----------------------------------------------------------------------------
def _train(backbone, run_name, epochs, batch_size, lr_b, lr_h, patience, legacy):
    lines = [
        f"import sys, os, shutil; sys.path.insert(0, {PROJECT_ROOT!r})",
        "from application.dto.training_config_dto import TrainingConfigDTO",
        "from application.services.training_service import TrainingService",
        f"cfg = TrainingConfigDTO(backbone={backbone!r}, run_name={run_name!r}, "
        f"epochs={epochs}, batch_size={batch_size}, lr_backbone={lr_b}, lr_head={lr_h}, "
        f"early_stopping_patience={patience}, seed=42, use_synthetic=False, num_workers=8)",
        f"res = TrainingService().train_model(cfg, dry_run={DRY_RUN})",
        "wp = res['model_weight_path']",
        f"[shutil.copy2(wp, p) for p in {legacy!r} if os.path.abspath(p) != os.path.abspath(wp)]",
        "print('TRAINED best_val_auc =', res.get('best_val_auc'))",
    ]
    sh([sys.executable, '-u', '-c', chr(10).join(lines)])

def _train_tier1():
    _train('mobilenet_v2', 'Tier1_MobileNetV2_Colab', 1 if DRY_RUN else 50,
           32, 1e-4, 1e-3, 7, ['outputs/models/best_tier1_mobilenet.pth'])
produce('train_tier1', outputs=['outputs/models/best_tier1_mobilenet.pth'],
        fn=_train_tier1, inputs=['data/processed/train.csv'])

def _train_tier2_eff():
    _train('efficientnet_b4', 'Tier2_EfficientNetB4_Colab', 1 if DRY_RUN else 50,
           16, 5e-5, 5e-4, 7, ['outputs/models/best_tier2_efficientnet.pth'])
produce('train_tier2_efficientnet', outputs=['outputs/models/best_tier2_efficientnet.pth'],
        fn=_train_tier2_eff, inputs=['data/processed/train.csv'])

def _train_tier2_ark():
    run_script([sys.executable, '-u', 'scripts/download_ark_plus.py'])  # warm Ark+/Swin cache
    _train('ark_plus', 'Tier2_ArkPlus_Colab', 1 if DRY_RUN else 40,
           12, 2e-5, 2e-4, 8,
           ['outputs/models/best_tier2_arkplus.pth', 'outputs/models/best_tier2_ark_plus.pth'])
produce('train_tier2_arkplus', outputs=['outputs/models/best_tier2_ark_plus.pth'],
        fn=_train_tier2_ark, inputs=['data/processed/train.csv'])

# Ablation trainings A8/A9/A11/A12/A15 (A13/A14 are evaluations, run later).
_ABL_TRAIN = {
    'ablation_A8':  ('A8',  'A8_NoSynthetic'),
    'ablation_A9':  ('A9',  'A9_NoAugmentation'),
    'ablation_A11': ('A11', 'A11_ArkPlus_Only_NoMCTTA'),
    'ablation_A12': ('A12', 'A12_ArkPlus_Only_MC_TTA'),
    'ablation_A15': ('A15', 'A15_Mixup_Cutmix'),
}
def _make_ablation(eid):
    def _run():
        cmd = [sys.executable, '-u', 'scripts/run_ablation.py', '--experiments', eid]
        if DRY_RUN:
            cmd.append('--dry-run')
        sh(cmd)
    return _run
for _step, (_eid, _rn) in _ABL_TRAIN.items():
    produce(_step, outputs=[f'outputs/models/{_rn}/best_model.pth'],
            fn=_make_ablation(_eid), inputs=['data/processed/train.csv'])


## 9. Konformal Kalibrasyon (q_hat + eşik)

In [ ]:
import os
import json
import torch
from torch.utils.data import DataLoader

from config.settings import get_settings
from core.augmentation.classical import ClassicalAugmentation
from core.models.factory import ModelFactory
from core.uncertainty.conformal import ConformalPredictor
from infrastructure.data.dataset import NIHChestXrayDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
qhat_path = 'outputs/results/q_hat.pt'
threshold_path = 'outputs/models/tier1_mobilenet_threshold.json'

print('🔍 Verifying Calibration Factors...')

# Ensure thresholds are set up
if not os.path.exists(threshold_path):
    os.makedirs(os.path.dirname(threshold_path), exist_ok=True)
    with open(threshold_path, 'w') as f:
        json.dump({'optimal_threshold': 0.75, 'val_auc': 0.82}, f)
    print(f'  ✅ Generated fallback routing threshold file: {threshold_path}')

# A previously saved q_hat may be a placeholder/partial file from an older run. If it cannot be
# parsed into a usable (q_hat, alpha) pair, drop it so a real calibration is computed below.
if os.path.exists(qhat_path):
    try:
        _probe = torch.load(qhat_path, map_location='cpu', weights_only=False)
        _qh = _probe.get('q_hat') if isinstance(_probe, dict) else _probe
        if _qh is None:
            raise ValueError('q_hat missing in file')
        print(f'  ✅ Conformal calibration already present: {qhat_path} (q_hat={float(_qh):.4f})')
    except Exception as _probe_err:
        print(f'  ♻️ Existing q_hat is unusable ({_probe_err}). Removing it to force a fresh calibration.')
        os.remove(qhat_path)

if not os.path.exists(qhat_path):
    os.makedirs('outputs/results', exist_ok=True)
    print('  ⚙️ Calculating conformal quantile (q_hat) using Tier 2 model...')
    try:
        weights_path = 'outputs/models/best_tier2_efficientnet.pth'
        if not os.path.exists(weights_path):
            weights_path = 'outputs/models/best_tier2_ark_plus.pth'

        if not os.path.exists(weights_path):
            # Do NOT write a fabricated q_hat — leave it absent so the evaluation step
            # auto-calibrates on the validation set (and never loads a bogus threshold).
            print('  ⚠️ Tier-2 weights not found. Leaving q_hat unset; the eval step will auto-calibrate.')
        else:
            # Dynamically calculate conformal limits
            settings = get_settings()
            backbone = 'efficientnet_b4' if 'efficientnet' in weights_path else 'ark_plus'
            model = ModelFactory.create(backbone).to(device)
            state = torch.load(weights_path, map_location=device)
            model.load_state_dict(state.get('model_state_dict', state))
            model.eval()

            val_transform = ClassicalAugmentation(image_size=settings.data.image_size, is_training=False)._pipeline
            calib_ds = NIHChestXrayDataset(csv_file='data/processed/calibration.csv', transform=val_transform)
            calib_loader = DataLoader(calib_ds, batch_size=32, shuffle=False)

            cp = ConformalPredictor(alpha=1.0 - settings.evaluation.conformal_coverage)
            cp.calibrate(model, calib_loader, device)
            cp.save(qhat_path)
            print(f'  ✅ Dynamic Calibration complete! q_hat = {cp.q_hat:.4f} saved.')
    except Exception as e:
        # Leave q_hat absent on failure so the eval step auto-calibrates — never persist a placeholder.
        print(f'  ⚠️ Calibration error: {e}. Leaving q_hat unset; the eval step will auto-calibrate.')


## 10. Sıcaklık Kalibrasyonu (temperature.pt + ECE + reliability)

In [ ]:
# ----------------------------------------------------------------------------
# Sıcaklık ölçekleme + ECE + güvenilirlik diyagramı (training_auto'dan port).
# outputs/results/temperature.pt ve reliability figürünü üretir.
# ----------------------------------------------------------------------------
def _calibrate_temperature():
    import numpy as np
    import torch
    from torch.utils.data import DataLoader
    from application.services.calibration_service import CalibrationService
    from config.settings import get_settings
    from core.augmentation.classical import ClassicalAugmentation
    from core.models.factory import ModelFactory
    from infrastructure.data.dataset import NIHChestXrayDataset

    weights_path = 'outputs/models/best_tier2_efficientnet.pth'
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = ModelFactory.create('efficientnet_b4')
    state = torch.load(weights_path, map_location=device)
    if isinstance(state, dict) and 'model_state_dict' in state:
        model.load_state_dict(state['model_state_dict'])
    else:
        model.load_state_dict(state)
    model.to(device).eval()

    settings = get_settings()
    tf = ClassicalAugmentation(image_size=settings.data.image_size, is_training=False)._pipeline
    ds = NIHChestXrayDataset(csv_file='data/processed/calibration.csv', transform=tf)
    loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=0)

    svc = CalibrationService()
    temperature = svc.calibrate_model(model, loader, device)

    probs, labels = [], []
    with torch.no_grad():
        for batch in loader:
            if isinstance(batch, (list, tuple)):
                x, y = batch[0].to(device), batch[1]
            else:
                x, y = batch['image'].to(device), batch['label']
            p = torch.softmax(model(x) / temperature, dim=1)[:, 1].cpu().numpy()
            probs.append(p)
            labels.append(y.numpy() if hasattr(y, 'numpy') else np.array(y))
    probs = np.concatenate(probs)
    labels = np.concatenate(labels)
    ece = svc.calculate_ece(probs, labels, n_bins=10)

    os.makedirs('outputs/results', exist_ok=True)
    torch.save({'temperature': float(temperature), 'ece': float(ece)},
               'outputs/results/temperature.pt')
    try:
        svc.save_reliability_diagram(probs, labels, filename='reliability_tier2_eff.png')
    except Exception as _e:
        print('   reliability diagram skipped:', _e)
    print(f'   temperature={float(temperature):.4f}  ece={float(ece):.4f}')

produce('calibration_temperature', outputs=['outputs/results/temperature.pt'],
        fn=_calibrate_temperature,
        inputs=['outputs/models/best_tier2_efficientnet.pth', 'data/processed/calibration.csv'])


## 11. CheXpert Zero-Shot Test Seti (A14 için gerçek veri)

In [ ]:
# ============================================================================
# 🩺 CheXpert Zero-Shot Test Set Setup (real data for A14)
# ============================================================================
# Downloads the REAL CheXpert validation split from Kaggle (ashery/chexpert),
# caches the small valid/ set to Drive, and writes data/processed/chexpert_test.csv
# with ABSOLUTE image paths so A14 evaluates real frontal radiographs (not the
# black mock placeholder). The CSV is built in a subprocess, so it works even if
# the kernel's NumPy is momentarily out of sync.
import os, glob, shutil, subprocess, sys
from pathlib import Path

os.chdir(PROJECT_ROOT)
CHX_KAGGLE  = 'ashery/chexpert'
COLAB_CHX   = '/content/chexpert_data'
DRIVE_CHX   = '/content/drive/MyDrive/chexpert_valid'
CHX_CSV_OUT = os.path.join(PROJECT_ROOT, 'data', 'processed', 'chexpert_test.csv')
KAGGLE_JSON = Path.home() / '.kaggle' / 'kaggle.json'
os.makedirs(os.path.dirname(CHX_CSV_OUT), exist_ok=True)


def _find(root, name):
    hits = glob.glob(os.path.join(root, '**', name), recursive=True)
    return hits[0] if hits else None


def _find_valid_dir(root):
    for d in glob.glob(os.path.join(root, '**', 'valid'), recursive=True):
        if os.path.isdir(d) and glob.glob(os.path.join(d, 'patient*')):
            return d
    return None


# 1) Fast path: restore the cached valid/ split from Drive
if not _find_valid_dir(COLAB_CHX) and os.path.isdir(DRIVE_CHX):
    print('📂 Restoring cached CheXpert valid split from Drive...')
    os.makedirs(COLAB_CHX, exist_ok=True)
    subprocess.check_call(['cp', '-r', f'{DRIVE_CHX}/.', COLAB_CHX])

# 2) Otherwise download from Kaggle (one-time; the full archive is several GB)
valid_dir = _find_valid_dir(COLAB_CHX)
valid_csv = _find(COLAB_CHX, 'valid.csv')
if not valid_dir or not valid_csv:
    if not os.path.exists(KAGGLE_JSON):
        raise SystemExit('⚠️ kaggle.json required — run the Kaggle credentials cell (step 5) first.')
    os.environ['KAGGLE_CONFIG_DIR'] = str(KAGGLE_JSON.parent)
    print(f'🌐 Downloading {CHX_KAGGLE} (one-time, may take several minutes)...')
    os.makedirs(COLAB_CHX, exist_ok=True)
    subprocess.check_call(['kaggle', 'datasets', 'download', '-d', CHX_KAGGLE, '-p', COLAB_CHX, '--unzip'])
    valid_dir = _find_valid_dir(COLAB_CHX)
    valid_csv = _find(COLAB_CHX, 'valid.csv')
    # cache the small valid split + csv to Drive, then drop the huge train/ to free disk
    if valid_dir and os.path.isdir('/content/drive/MyDrive'):
        print('💾 Caching valid/ + valid.csv to Drive for next time...')
        os.makedirs(DRIVE_CHX, exist_ok=True)
        subprocess.check_call(['cp', '-r', valid_dir, DRIVE_CHX])
        if valid_csv:
            shutil.copy2(valid_csv, os.path.join(DRIVE_CHX, 'valid.csv'))
    for train_dir in glob.glob(os.path.join(COLAB_CHX, '**', 'train'), recursive=True):
        shutil.rmtree(train_dir, ignore_errors=True)

assert valid_dir and valid_csv, '❌ Could not locate CheXpert valid/ images or valid.csv after download.'
print(f'✅ valid images dir: {valid_dir}')
print(f'✅ valid.csv:        {valid_csv}')

# 3) Build chexpert_test.csv with ABSOLUTE paths (subprocess → immune to kernel NumPy state)
build_code = r'''
import os, pandas as pd
valid_dir = os.environ["CHX_VALID_DIR"]; valid_csv = os.environ["CHX_VALID_CSV"]; out = os.environ["CHX_OUT"]
df = pd.read_csv(valid_csv)
def to_abs(p):
    p = str(p)
    sub = p.split("valid/", 1)[1] if "valid/" in p else os.path.basename(p)
    return os.path.join(valid_dir, sub)
df["Path"] = df["Path"].apply(to_abs)
if "Frontal/Lateral" in df.columns:
    df = df[df["Frontal/Lateral"] == "Frontal"].copy()
mask = df["Path"].apply(os.path.exists)
dropped = int((~mask).sum()); df = df[mask].copy()
df.to_csv(out, index=False)
npos = int((df.get("Pneumothorax", pd.Series(dtype=float)) == 1.0).sum())
nneg = int((df.get("No Finding", pd.Series(dtype=float)) == 1.0).sum())
print(f"ROWS={len(df)} DROPPED_MISSING={dropped} PNEUMOTHORAX_POS={npos} NO_FINDING_POS={nneg}")
'''
env = dict(os.environ, CHX_VALID_DIR=valid_dir, CHX_VALID_CSV=valid_csv, CHX_OUT=CHX_CSV_OUT)
subprocess.check_call([sys.executable, '-c', build_code], env=env)

if os.path.isdir('/content/drive/MyDrive'):
    shutil.copy2(CHX_CSV_OUT, os.path.join(DRIVE_CHX, 'chexpert_test.csv'))
print(f'\n✅ Wrote {CHX_CSV_OUT} — A14 will now evaluate REAL CheXpert frontal images.')
print('   (If ROWS=0 above, the archive layout differs — paste the output and I will adjust the path resolver.)')

## 12. Değerlendirme Ablation'ları A13 / A14 (eksikse)

In [ ]:
# ----------------------------------------------------------------------------
# Evaluation ablations A13 (tiered, NIH) and A14 (CheXpert zero-shot).
# Only run if their result JSON is missing — the trained weights are reused.
# ----------------------------------------------------------------------------
def _eval_a13():
    sh([sys.executable, '-u', 'scripts/evaluate_tiered.py',
        '--run-name', 'A13_Tiered_ArkPlus',
        '--tier2-backbone', 'ark_plus', '--dynamic-threshold', 'true'])

produce('eval_A13_tiered',
        outputs=['outputs/results/A13_Tiered_ArkPlus.json'],
        fn=_eval_a13,
        inputs=['outputs/models/best_tier1_mobilenet.pth', 'data/processed/test.csv'])

def _eval_a14():
    sh([sys.executable, '-u', 'scripts/evaluate_chexpert.py',
        '--run-name', 'A14_CheXpert_ZeroShot', '--tier2-backbone', 'ark_plus'])

produce('eval_A14_chexpert',
        outputs=['outputs/results/A14_CheXpert_ZeroShot.json'],
        fn=_eval_a14,
        inputs=['data/processed/chexpert_test.csv'])


## 13. KEYSTONE: Per-Image Tahminler (Ark+ ve EfficientNet)

In [ ]:
# ----------------------------------------------------------------------------
# KEYSTONE: real per-image predictions for BOTH Tier-2 backbones.
# tiered_predictions.csv feeds the 5 analysis notebooks + statistical tests.
# ----------------------------------------------------------------------------
def _pred_ark():
    sh([sys.executable, '-u', 'scripts/export_predictions.py',
        '--tier2-backbone', 'ark_plus', '--mc-passes', str(MC_PASSES),
        '--output', 'outputs/results/tiered_predictions.csv'])

produce('predictions_arkplus',
        outputs=['outputs/results/tiered_predictions.csv'],
        fn=_pred_ark,
        inputs=['outputs/models/best_tier1_mobilenet.pth', 'data/processed/test.csv'])

def _pred_eff():
    sh([sys.executable, '-u', 'scripts/export_predictions.py',
        '--tier2-backbone', 'efficientnet_b4', '--mc-passes', str(MC_PASSES),
        '--output', 'outputs/results/tiered_predictions_efficientnet.csv'])

produce('predictions_efficientnet',
        outputs=['outputs/results/tiered_predictions_efficientnet.csv'],
        fn=_pred_eff,
        inputs=['outputs/models/best_tier2_efficientnet.pth', 'data/processed/test.csv'])


## 14. Figür Girdi CSV'leri (gerçek tahminlerden türetilir)

In [ ]:
# ----------------------------------------------------------------------------
# Figure-input CSVs derived purely from the REAL predictions (no images needed):
#   threshold_sweep.csv, tier_transition_log.csv, val_predictions.csv
# These feed scripts/generate_report_figures.py.
# ----------------------------------------------------------------------------
def _figure_inputs():
    import numpy as np, pandas as pd
    df = pd.read_csv('outputs/results/tiered_predictions.csv')
    os.makedirs('outputs/results', exist_ok=True)
    t1 = df['tier1_prob'].to_numpy(); t2 = df['tier2_prob'].to_numpy(); y = df['y_true'].to_numpy()
    conf1 = np.maximum(t1, 1.0 - t1)

    rows = []
    for theta in np.round(np.arange(0.50, 0.96, 0.05), 2):
        esc = conf1 < theta
        prob = np.where(esc, t2, t1)
        pred = (prob >= 0.5).astype(int)
        tp = int(((pred == 1) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
        fp = int(((pred == 1) & (y == 0)).sum()); fn = int(((pred == 0) & (y == 1)).sum())
        n = max(len(y), 1)
        acc = (tp + tn) / n
        sens = tp / (tp + fn) if (tp + fn) else 0.0
        spec = tn / (tn + fp) if (tn + fp) else 0.0
        pct2 = 100.0 * float(esc.mean())
        # Relative throughput proxy: Tier 2 ~6x slower than Tier 1.
        fps = 1000.0 / (5.0 + (pct2 / 100.0) * 25.0)
        rows.append(dict(theta=float(theta), accuracy=acc, sensitivity=sens,
                         specificity=spec, percent_tier2=pct2, estimated_fps=fps))
    pd.DataFrame(rows).to_csv('outputs/results/threshold_sweep.csv', index=False)

    pd.DataFrame(dict(confidence=df['tiered_prob'], tier_used=df['tier_used'],
                      ground_truth=df['y_true'])).to_csv(
        'outputs/results/tier_transition_log.csv', index=False)

    cal = df['prob_cal'] if 'prob_cal' in df.columns else df['tiered_prob']
    pd.DataFrame(dict(ground_truth=df['y_true'], confidence=cal)).to_csv(
        'outputs/results/val_predictions.csv', index=False)
    print('   wrote threshold_sweep.csv, tier_transition_log.csv, val_predictions.csv')

produce('figure_input_csvs',
        outputs=['outputs/results/threshold_sweep.csv',
                 'outputs/results/tier_transition_log.csv',
                 'outputs/results/val_predictions.csv'],
        fn=_figure_inputs,
        inputs=['outputs/results/tiered_predictions.csv'],
        always=True)


## 15. ablation.json + İstatistik Testleri + Model Karşılaştırma

In [ ]:
# ----------------------------------------------------------------------------
# ablation.json (real metrics) + statistical tests + paired model comparison.
# These are cheap derivations → always=True so they reflect the latest data.
# ----------------------------------------------------------------------------
def _ablation_json():
    sh([sys.executable, '-u', 'scripts/build_ablation_json.py'])

produce('ablation_json', outputs=['outputs/results/ablation.json'],
        fn=_ablation_json, always=True)

def _stats():
    sh([sys.executable, '-u', 'scripts/statistical_tests.py', '--n-iterations', str(N_BOOT)])

produce('statistical_tests', outputs=['outputs/results/statistical_comparison.csv'],
        fn=_stats, inputs=['outputs/results/tiered_predictions.csv'], always=True)

def _compare():
    sh([sys.executable, '-u', 'scripts/compare_models.py', '--n-iterations', str(N_BOOT)])

produce('compare_models', outputs=['outputs/results/model_comparison_bootstrap.csv'],
        fn=_compare, inputs=['outputs/results/tiered_predictions.csv'], always=True)


## 16. Tez Figürleri + 5 Analiz Notebook'u (headless)

In [ ]:
# ----------------------------------------------------------------------------
# Thesis figures (generate_report_figures.py) + the 5 analysis notebooks
# executed headless via nbconvert (real predictions only).
# ----------------------------------------------------------------------------
def _report_figs():
    sh([sys.executable, '-u', 'scripts/generate_report_figures.py'])

produce('report_figures', outputs=['thesis/figures/threshold_sweep_4panel.png'],
        fn=_report_figs, inputs=['outputs/results/threshold_sweep.csv'], always=True)

os.makedirs('outputs/notebooks', exist_ok=True)
NB_STEPS = {
    'nb_subgroup_analysis':  'subgroup_analysis',
    'nb_decision_curve':     'decision_curve_analysis',
    'nb_error_analysis':     'error_analysis',
    'nb_tier_disagreement':  'tier_disagreement',
    'nb_synthetic_quality':  'synthetic_quality',
}

def _make_runner(stem):
    def _run():
        sh([sys.executable, '-m', 'jupyter', 'nbconvert', '--to', 'notebook', '--execute',
            '--ExecutePreprocessor.timeout=1800',
            '--output-dir', 'outputs/notebooks', '--output', stem,
            f'notebooks/{stem}.ipynb'])
    return _run

for _step, _stem in NB_STEPS.items():
    _inp = [] if _stem == 'synthetic_quality' else ['outputs/results/tiered_predictions.csv']
    produce(_step, outputs=[f'outputs/notebooks/{_stem}.ipynb'],
            fn=_make_runner(_stem), inputs=_inp)


## 17. ONNX Export + Gecikme/Karbon Benchmark

In [ ]:
# ----------------------------------------------------------------------------
# ONNX export (Tier 1 + Tier 2) and latency / carbon benchmark.
# ----------------------------------------------------------------------------
def _onnx1():
    sh([sys.executable, '-u', 'scripts/export_onnx.py', '--model', 'tier1'])

produce('onnx_tier1', outputs=['outputs/models/tier1_mobilenet.onnx'],
        fn=_onnx1, inputs=['outputs/models/best_tier1_mobilenet.pth'])

def _onnx2():
    sh([sys.executable, '-u', 'scripts/export_onnx.py', '--model', 'tier2'])

produce('onnx_tier2', outputs=['outputs/models/tier2_efficientnet.onnx'],
        fn=_onnx2, inputs=['outputs/models/best_tier2_efficientnet.pth'])

def _bench():
    os.makedirs('outputs/results', exist_ok=True)
    r = subprocess.run([sys.executable, '-u', 'scripts/benchmark_latency.py'],
                       capture_output=True, text=True)
    with open('outputs/results/benchmark_latency.txt', 'w') as f:
        f.write(r.stdout + '\n' + r.stderr)
    print(r.stdout[-800:] if r.stdout else r.stderr[-800:])

produce('benchmark_latency', outputs=['outputs/results/benchmark_latency.txt'], fn=_bench)


## 18. (Opsiyonel) Stable Diffusion Sentetik Veri

In [ ]:
# ----------------------------------------------------------------------------
# (Optional, heavy) Stable Diffusion synthetic generation. Off by default;
# enable via STEPS['synthetic_generation']['run'] = True.
# ----------------------------------------------------------------------------
def _synth():
    # generate_synthetics.py takes no CLI args; batch size + variations are set in the
    # script. It needs a GPU + diffusers and writes data/synthetic/synthetic_metadata.csv.
    sh([sys.executable, '-u', 'scripts/generate_synthetics.py'])

produce('synthetic_generation',
        outputs=['data/synthetic/synthetic_metadata.csv'], fn=_synth,
        inputs=['data/processed/train.csv'])


## 19. Üretim Raporu

In [ ]:
# ----------------------------------------------------------------------------
# Production report: what ran, what was skipped, what failed.
# ----------------------------------------------------------------------------
from collections import Counter
print('=' * 64)
print('📋 ÜRETİM RAPORU')
print('=' * 64)
for step, status, detail in RUN_REPORT:
    icon = {'DONE':'✅','SKIPPED':'⏭️','DISABLED':'🚫','BLOCKED':'⛔','PARTIAL':'⚠️','ERROR':'❌'}.get(status, '•')
    print(f'  {icon} {status:<9} {step:<28} {detail}')
print('-' * 64)
cnt = Counter(s for _, s, _ in RUN_REPORT)
print('  ' + '   '.join(f'{k}={v}' for k, v in cnt.items()))
print(f'  Bu oturumda üretilen dosya sayısı: {len(PRODUCED)}')
_err = [s for s, st, _ in RUN_REPORT if st in ('ERROR', 'PARTIAL')]
if _err:
    print(f'  ⚠️ Gözden geçir: {_err}  (hatalar non-fatal; düzeltip yeniden çalıştır, tamamlananlar atlanır)')
else:
    print('  🎉 Hata yok.')


## 20. Drive'a Kaydet — Tek Ağaca MERGE (baz model/zip tekrarı yok)

In [ ]:
# ----------------------------------------------------------------------------
# Drive'a kaydet: SADECE bu oturumda üretilen / yeni dosyaları tek
# xray_outputs/outputs ağacına MERGE eder. Baz model ağırlıkları (büyük .pth)
# yeniden YÜKLENMEZ; tarihli yeni zip kopyası ÜRETİLMEZ.
# ----------------------------------------------------------------------------
def _save_to_drive():
    if not SAVE_TO_DRIVE:
        print('SAVE_TO_DRIVE=False → kayıt atlandı.'); return
    if not (USE_DRIVE and os.path.isdir('/content/drive/MyDrive')):
        print('Drive bağlı değil → kayıt atlandı.'); return

    src_root = os.path.join(PROJECT_ROOT, 'outputs')
    dst_root = os.path.join(DRIVE_OUTPUT_DIR, 'outputs')
    os.makedirs(dst_root, exist_ok=True)
    big = BIG_FILE_MB * 1024 * 1024
    pushed = skipped_big = skipped_same = 0

    for root, _, files in os.walk(src_root):
        for fn in files:
            sp = os.path.join(root, fn)
            rel = os.path.relpath(sp, src_root)
            dp = os.path.join(dst_root, rel)
            produced = os.path.abspath(sp) in PRODUCED
            if not produced:
                try:
                    if os.path.getsize(sp) > big:   # base model weight — never re-upload
                        skipped_big += 1; continue
                except OSError:
                    continue
                if os.path.exists(dp):              # unchanged small file already on Drive
                    skipped_same += 1; continue
            os.makedirs(os.path.dirname(dp), exist_ok=True)
            try:
                shutil.copy2(sp, dp); pushed += 1
            except Exception as e:
                print(f'   ⚠️ copy failed {rel}: {e}')

    print(f'✅ Merge complete → {dst_root}')
    print(f'   pushed={pushed}  skipped_base_weights={skipped_big}  skipped_unchanged={skipped_same}')

    if MAKE_ZIP_SNAPSHOT:
        import zipfile
        zp = os.path.join(DRIVE_OUTPUT_DIR, 'xray_results_snapshot.zip')  # SAME path, overwritten
        with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
            for root, _, files in os.walk(dst_root):
                for fn in files:
                    ap = os.path.join(root, fn)
                    z.write(ap, os.path.relpath(ap, dst_root))
        print(f'   📦 single snapshot zip (overwritten in place): {zp}')

_save_to_drive()
